In [1]:
! nvidia-smi

Tue Mar 10 07:43:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install spacy openpyxl

In [3]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 89.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
import pandas as pd
import re
import spacy
import csv
import openpyxl
import numpy as np
import nltk

In [5]:
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer # import WordNetLemmatizer

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


In [6]:
patent_data = pd.read_excel('氫燃料電池美國核准專利.xlsx')
test1 = patent_data['Title - DWPI'].fillna('')
test2 = patent_data['Abstract - DWPI'].fillna('')
test = test1 + ' ' + test2


In [7]:
nlp = spacy.load('en_core_web_sm')
spacy_stopwords = spacy.lang.en.stop_words.STOP_WORDS
stop_words_file_path = 'custom_stop_words.xlsx'  # 替換為你的 Excel 文件路徑
custom_stop_words_df = pd.read_excel(stop_words_file_path, header=None)

# 將 Excel 文件中的停用字轉換為列表
excel_stop_words = custom_stop_words_df[0].tolist()

# Remove any NaN values from the list of custom stop words
excel_stop_words = [word for word in excel_stop_words if str(word) != 'nan']

# 將合併後的停用字擴展到 spaCy 的停用字集合中
spacy_stopwords.update(excel_stop_words)
print(f"Total stop words: {len(spacy_stopwords)}")
for word in spacy_stopwords:
  if type(word) is str: # Check if the word is a string before adding to vocabulary
    nlp.vocab[word].is_stop = True
print("Custom stop words have been added successfully.")

# Convert the set of stop words to a list
stop_words_list = list(spacy_stopwords)
print(spacy_stopwords)

Total stop words: 28122
Custom stop words have been added successfully.
{'microprocessors', 'inhibition', 'unit80Load', 'switchQ3Third', 24, 25, 'stack100Controller', 'stack21Stack', '380', 36, 'Independent', 'heater34Fuel', 'stack11Cell', 'transversal', 67, 'propellers', 'algorithms', 'hydride11Storage', 'F2Main', 'water40Freezing', '254', 'portion368Valve', 'AAOutput', 'prepare', 111, 'anode84Step', 'degassing', 'controller66Accelerator', '79Wheels81Steering', '6Control', 'Between', 'softening', "'s", 'his', '144side', 'controller144Electrical', '2Cryogenic', 'tangent', 'compartment103Battery104Kiosks', 'programmable', 'heateran', 'vacuum', 'Also', 'device15Valve20Programmable', 'cyanide', 'walls', 'deployment', 'system23', '17Three', 'converter22Capacitor23Inverter24Motor100Electric', '14bEnd', 'electrode12Liquid', 'Speed', 'layer18Flow', 'vs', 'body24Sensor', 'O3', '2Width', 'inlet30Pumping', 'vehicle32Fill', '10Solid', 'DCDC', 'membrane22', 'deforms', 'Polyelemental', 'acrylics', 

In [8]:
# 載入 spaCy 英文模型
lemmatizer = WordNetLemmatizer()
# 獲取單詞的詞性
def get_wordnet_pos(pos_tag):
    """Map POS tag to first character lemmatize() accepts"""
    tag_dict = {"ADJ": wordnet.ADJ,
                "NOUN": wordnet.NOUN,
                "VERB": wordnet.VERB,
                "ADV": wordnet.ADV}
    return tag_dict.get(word, wordnet.NOUN)

def preprocess_text(text):
    if pd.isna(text):
        return ""

     # 🔹 分離數字與英文黏詞（如 13Valve → Valve）
    if re.search(r'\d+[A-Za-z]', text):
        text_parts = re.split(r'(\d+[A-Za-z]+)', text)
        text = ' '.join(
            re.sub(r'\d+', '', part) if re.match(r'\d+[A-Za-z]+', part) else part
            for part in text_parts
        )
    # 🔹 分句
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    # 規則：圖示說明句的開頭
    illustration_patterns = [
        r'^the (drawing|figure|diagram) show(s)?',]

    # 過濾圖示說明句
    filtered_sentences = [
        s for s in sentences
        if not any(re.match(pat, s.strip(), re.I) for pat in illustration_patterns)
    ]
    # 🔹 合併回剩下句子
    cleaned_text = ' '.join(filtered_sentences)

    # 小寫 + 去標點 + 去數字
    cleaned_text = re.sub(r'[^\w\s]', '', cleaned_text.lower())
    cleaned_text = re.sub(r'\d+', '', cleaned_text)

    # spaCy 處理
    doc = nlp(cleaned_text)
    tokens = [
        lemmatizer.lemmatize(token.text.lower(), get_wordnet_pos(token.pos_))
        for token in doc
        if token.is_alpha and token.text.lower() not in stop_words_list
    ]

    return ' '.join(tokens)

In [9]:
# 預處理 test 中的每一句文本
preprocessed_test = [preprocess_text(t) for t in test]

# 如果你想存回 DataFrame 中一起看
patent_data["Document"] = test
patent_data["Processed"] = preprocessed_test

# 顯示結果
print(patent_data[["Document", "Processed"]].head())

# 如果需要儲存
patent_data.to_excel("preprocessed_document.xlsx", index=False)

                                            Document  \
0  Method for operating fuel cell system, involve...   
1  Method for operating thermal management system...   
2  Air supply system for fuel cell, comprises ele...   
3  Cooling system for a fuel cell onboard a vehic...   
4  Method of extending the range of an electrical...   

                                           Processed  
0  method operating thermally coupling cooling ci...  
1  method operating thermal management eg train d...  
2  air supply comprises voltage source voltage ca...  
3  cooling onboard comprises configured liquid ga...  
4  method extending range electrically powered au...  
